# 01 Lecture notes

Where this work starts: what the Data Science Toolbox lectures teach that bears on Sprint 1. These are my notes in
my own words, from blocks 01 to 05 of the [coursebook](https://dsbristol.github.io/dst/coursebook.html) (Lawson,
2026), the blocks taught before the deadline. Each section ends with **what it means for us**, which feeds the plan
in `02-plan.ipynb`.

The brief ([Assessment 1](https://dsbristol.github.io/dst/assets/assessments/Assessment1.pdf)) asks each of us for
one model scored on the same left-out test data, and puts half the effort on the performance metric: when does it
say something about real-world generalisation, and is our best model truly best?

## Block 01: classification

*Lecture 01.2 Introduction to Classification; Portfolio 1.2.P Metrics.*

* Supervised learning splits the data three ways: **train** (fit parameters), **calibration** (choose
  hyperparameters), **test** (report performance). The test set scores the model; it must never steer it.
* **Logistic regression is the standard baseline** ("the go-to straw man classifier"): a natural predictive model
  that does surprisingly well. Its output is the log-odds, so it ranks cases and can be thresholded.
* Performance at one threshold is a confusion matrix (TP, FP, FN, TN). Across all thresholds:
  * the **ROC curve** (true positive rate against false positive rate) and its **AUC**, which hold whatever the class
    balance;
  * the **precision-recall curve**, better when we care specifically about the positive (rare) class.
* **Accuracy misleads** with imbalanced classes (always guessing the majority class scores well) and ignores
  uncertainty, so it can reward over-confident models.
* **Proper scoring rules** (log-likelihood, Brier score $\frac1n\sum_i (y_i-\hat p_i)^2$) are optimised only by
  reporting the true probability, so they reward honest probabilities: "keep probabilities in the model"
  (Gneiting & Raftery, 2007).
* If a metric is used to choose a model on the test data, a further test set is needed to test that choice.

**What it means for us:** our data is imbalanced (22% default), so no accuracy. Report AUC with ROC **and** PR
curves for ranking, Brier and log loss for probabilities, and check calibration. Tune on training data only.

## Block 02: regression, model selection, cross-validation

*Lecture 02.1 Modern Regression; Portfolio 2.1.P Model Selection; Lecture 02.2 Cross Validation; Portfolio 2.2.P
Leave-one-out.*

**Regression in matrix form.** With a design matrix $X$ (a column of ones, then one column per feature; categories
one-hot encoded) least squares gives $\hat\beta = (X^\top X)^{-1}X^\top y$. Each coefficient is read against its
standard error ($t = \hat\beta_j/\hat\sigma_j$). Regression is a *conditional* model of $y$ given $x$, and finding an
association is **not** causation. Logistic regression is the same linear predictor $X\beta$ on the log-odds scale,
fitted by maximum likelihood instead of least squares.

**Bigger models always fit the training data better.** For nested models the larger one has smaller training error
unless the extra column is useless. Out-of-sample error is larger than training error by an "optimism" that grows
with the number of parameters $p$. That motivates penalised criteria:

* AIC $= -2\log L(\hat\theta) + 2p$;
* BIC $= -2\log L(\hat\theta) + p\log n$, which penalises more as $n$ grows.

**Cross-validation.** Split the training data into $k$ folds; fit on $k-1$, score the left-out fold, average. $k = 10$
is the usual choice for large data. It can choose model structure; the test data is looked at once, at the end.

* **Correlated data:** folds must respect the correlation. One option is to *redefine what a data point is*, e.g.
  "countries instead of countries at time points", and leave out whole units.
* **Leave-one-out** is not enough unless the data are independent.
* **What are we generalising to?** The lecture lists four targets, each needing a different split:
  1. the next point in the same stream;
  2. a new segment of the same stream;
  3. a new but similar source;
  4. an unexpected, new kind of source.

**What it means for us:** a hazard model has one row per customer per month, so rows are correlated. The "data
point" is the **customer**: cross-validate with whole customers left out. Compare nested models with AIC, BIC and
cross-validation, and expect the training fit to always favour the bigger model. The four targets are a ready-made
frame for the metric half of the brief.

## Block 03: losses, trees and ensembles

*Lecture 03.1 Decision Trees; Portfolio 3.1.P; Lecture 03.2 Ensembles.*

* A model is chosen to minimise a **loss**; the loss defines "best". Trees split on Gini impurity or entropy
  (information), which weight mistakes differently.
* Trees overfit, so they are pruned with a penalty or with cross-validation. A single held-out set "risks
  overfitting"; random cross-validation sets reduce that at the cost of more fits.
* Bagging, random forests, boosting and stacking combine many models to reduce variance and bias.

**What it means for us:** trees are Louis's model. For us: choose penalties by cross-validation, not one validation
set, and stacking is the obvious answer to "how would we change the model to win?".

## Block 04: latent structure and clustering

*Lecture 04.1 Latent Structures and PCA; Lecture 04.2 Clustering; Portfolio 4.2.P Mixture Modelling.*

* **PCA / SVD** finds the main directions of variation. "Spectral clustering" is just clustering on the top
  principal components.
* Clustering families: algorithmic, distance-based (hierarchical: single, complete, average linkage) and
  model-based (k-means, Gaussian mixtures). The **choice of distance and scaling** matters.
* **k-means** alternates "assign each point to its nearest centre" and "move each centre to its members' mean". It
  assumes round, similar-sized clusters; use many random starts.
* **Gaussian mixture models (GMM)** allow elliptical clusters with soft membership, fitted by
  **expectation-maximisation (EM)**: alternately compute each point's cluster probabilities, then update the
  clusters. k-means is the special case of equal round clusters. The number of clusters and covariance shape can be
  chosen by **BIC**.
* **How many clusters, really?** If the clusters are meant to be real, careful model selection matters. If they are
  a tool for understanding, the number is a **tuning parameter**, and a **sensitivity analysis** should check
  whether it changes the conclusions.

**What it means for us:** our clusters are a tool (customer types to test the metric on), so run the analysis over a
range of K and report whether the conclusions move, rather than claiming one "true" K. Compare k-means with a GMM
chosen by BIC. EM is also the engine behind the mixture cure and latent-class hazard models in `../research/`.

## Block 05: testing and uncertainty

*Lecture 05.2 Towards Modern Statistical Testing; Portfolio 5.2.P Conformal Prediction. (Lecture 05.1 on anomaly
detection is less relevant here.)*

* Resampling methods: **bootstrap** (resample with replacement, for variability), **permutation** (resample without
  replacement, to test a null), **cross-validation** (out-of-sample performance).
* They need **exchangeable units**: if the rows are correlated (e.g. a time series), resampling rows is wrong.
* **Double dipping:** choosing a model on some data and then testing it on the same data gives over-optimistic
  results. Fix: select on one part, test on another.
* **Conformal prediction** wraps any model with a guarantee on held-out calibration data; the guarantee is
  *marginal* (on average), not for each subgroup. Calibrating separately within groups ("Mondrian") gives coverage
  within each group.

**What it means for us:** bootstrap **customers**, not customer-months. Tune by cross-validation on training data,
so the validation set is an honest check and the test set is used once. Results within customer groups (clusters)
can differ from the overall result, which is exactly what the brief asks us to examine.

## Summary: what the lectures tell us to do

| Lecture point | What we do |
|---|---|
| Logistic regression is the baseline (01.2) | The logit hazard is the core model; every variant is compared with it |
| ROC and PR curves; proper scoring rules (01.2, 1.2.P) | Report AUC, ROC and PR curves, Brier, log loss, calibration |
| Bigger models always fit better; AIC/BIC (2.1.P) | Nested hazard models compared with AIC, BIC and cross-validation |
| Cross-validate with correlated units left out together (02.2) | 10-fold cross-validation with whole customers left out |
| Four generalisation targets (02.2) | New customers, next month, new customer type |
| Choose penalties by cross-validation (3.1.P) | Penalty strength C chosen by grouped cross-validation |
| Number of clusters is a tuning parameter; GMM and BIC (04.2, 4.2.P) | Sensitivity analysis over K; k-means against GMM |
| Bootstrap exchangeable units; avoid double dipping (05.2) | Bootstrap customers; test set used once |

Next: `02-plan.ipynb` turns this into decisions.

**Reference:** Lawson, D. (2026). *Data Science Toolbox* [lecture notes, blocks 01 to 05]. University of Bristol.
https://dsbristol.github.io/dst/coursebook.html